<a href="https://colab.research.google.com/github/JSJeong-me/GPT-Web/blob/main/304_LangChain_DeepAgent_MCP_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangChain Deep Agent + LangChain Docs MCP

This Google Colab notebook connects a **Deep Agent** to the official **LangChain Docs MCP** and **LangChain API Reference MCP** servers.

Pipeline:

`User Question → Deep Agent → MCP Tools → LangChain Docs / API Reference → Final Answer`


## 1. Install required packages

Run this cell first. If Colab asks you to restart the runtime after installation, restart it and continue from the next cell.


In [1]:
!pip -q install -U deepagents "langchain[mcp]" langchain-openai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.7/859.7 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.4/242.4 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262

## 2. Set the OpenAI API key

The key is entered securely with `getpass`, so it is not displayed in notebook output.


In [2]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OPENAI_API_KEY: ")

print("OPENAI_API_KEY is configured.")


Enter your OPENAI_API_KEY: ··········
OPENAI_API_KEY is configured.


## 3. Define the Deep Agent with LangChain MCP tools

Two official MCP endpoints are connected:

- `https://docs.langchain.com/mcp` — guides, tutorials, patterns, and documentation
- `https://reference.langchain.com/mcp` — API reference and signatures


In [3]:
from deepagents import create_deep_agent
from langchain.messages import HumanMessage
from langchain.mcp import MCPAdapter


EXAMPLE_QUERY = "How do I stream intermediate tool results from a subagent?"


async def main(query: str = EXAMPLE_QUERY):
    mcp_config = {
        "mcpServers": {
            "docs-langchain": {
                "type": "http",
                "url": "https://docs.langchain.com/mcp",
            },
            "reference-langchain": {
                "type": "http",
                "url": "https://reference.langchain.com/mcp",
            },
        }
    }

    async with MCPAdapter(mcp_config) as adapter:
        tools = await adapter.list_tools()

        print("=== Available MCP Tools ===")
        for tool in tools:
            print(f"- {tool.name}")

        agent = create_deep_agent(
            model="openai:gpt-5.5",
            tools=tools,
            system_prompt=(
                "You are an expert LangChain and Deep Agents documentation assistant. "
                "Before answering technical questions, search the official LangChain "
                "documentation using the available MCP tools. Use the LangChain API "
                "Reference MCP tools when exact APIs, methods, parameters, signatures, "
                "or version-sensitive behavior are needed. Prefer current MCP-retrieved "
                "information over internal knowledge. Explain the relevant LangChain, "
                "LangGraph, and Deep Agents architecture when useful, and include concise "
                "Python examples when appropriate."
            ),
        )

        result = await agent.ainvoke(
            {"messages": [HumanMessage(content=query)]}
        )

        print("\n=== FINAL ANSWER ===\n")
        final_message = result["messages"][-1]
        final_text = getattr(final_message, "text", None)
        if final_text:
            print(final_text)
        else:
            print(final_message.content)

        return result


/tmp/ipykernel_891/794125225.py:3: LangChainBetaWarning: `langchain.mcp` is in beta. It is actively being worked on, so the API may change.
  from langchain.mcp import MCPAdapter


## 4. Run the example query

In Google Colab/Jupyter, top-level `await` is supported, so use `await main()` instead of `asyncio.run(main())`.


In [4]:
result = await main()


[09/10/26 02:05:43] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=556912;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=848538;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py#1384\1384]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

[09/10/26 02:05:44] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=984377;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=968166;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py#1384\1384]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

                    INFO     Proxy detected connected client - reusing existing session for all       ]8;id=245573;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=31288;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py#1384\1384]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

[09/10/26 02:05:45] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=25506;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=387340;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py#1384\1384]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

=== Available MCP Tools ===
- docs-langchain_search_docs_by_lang_chain
- docs-langchain_query_docs_filesystem_docs_by_lang_chain
- docs-langchain_submit_feedback
- reference-langchain_search_api
- reference-langchain_get_symbol

=== FINAL ANSWER ===

Use **Deep Agents event streaming** and subscribe to the subagent’s own `tool_calls` projection. Don’t expect the coordinator’s final `task` result to contain all intermediate tool outputs—the subagent is isolated and hands only a final report back to the main agent. The intermediate messages/tool calls are available on the stream.

## Python: stream subagent tool calls/results

```python
from deepagents import create_deep_agent
from langchain.tools import tool

@tool
def search_docs(query: str) -> str:
    """Search documentation."""
    return f"Results for {query}: ..."

agent = create_deep_agent(
    model="openai:gpt-5",
    tools=[],
    system_prompt=(
        "For research tasks, delegate to the researcher subagent using the task t

## 5. Run your own LangChain documentation query

Change `MY_QUERY` and execute the cell again.


In [5]:
MY_QUERY = "How can a Deep Agent delegate work to a subagent and stream its tool events?"

custom_result = await main(MY_QUERY)


[09/10/26 02:06:29] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=512548;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=909713;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py#1384\1384]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

                    INFO     Proxy detected connected client - reusing existing session for all       ]8;id=232656;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=699350;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py#1384\1384]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

                    INFO     Proxy detected connected client - reusing existing session for all       ]8;id=554179;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=970630;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py#1384\1384]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

[09/10/26 02:06:30] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=320947;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=779847;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py#1384\1384]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

=== Available MCP Tools ===
- docs-langchain_search_docs_by_lang_chain
- docs-langchain_query_docs_filesystem_docs_by_lang_chain
- docs-langchain_submit_feedback
- reference-langchain_search_api
- reference-langchain_get_symbol

=== FINAL ANSWER ===

A Deep Agent delegates synchronous sub-work through the built-in `task` tool. You make subagents available via `create_deep_agent(subagents=[...])`; the coordinator then calls `task` with `subagent_type=<subagent name>` and a `description`. To stream the delegated subagent’s tool events, use `stream_events(..., version="v3")` and read `subagent.tool_calls`.

```python
from deepagents import create_deep_agent


def search_docs(query: str) -> str:
    """Search internal docs."""
    return f"Results for: {query}"


agent = create_deep_agent(
    model="openai:gpt-5.5",
    system_prompt=(
        "You are a coordinator. For documentation research, delegate to "
        "the docs-researcher subagent using the task tool."
    ),
    subagents=

## Notes

- The MCP servers are accessed over the network, so Colab must have internet access.
- The OpenAI model name can be changed in `create_deep_agent()` if your account uses a different supported model.
- Keeping both Docs MCP and Reference MCP connected is useful when a question requires both conceptual guidance and exact API signatures.
